# Dataset Preparation Overview

This notebook is now a thin exploratory interface over the reusable pipeline modules in `pipeline.data`.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd

from pipeline.config import load_experiment_config
from pipeline.data.dataset import (
    build_dicom_mapping,
    create_augmentation_pipeline,
    load_metadata,
    normalize_and_resize_dicom,
    prepare_dataset,
)

experiment_config = load_experiment_config()
paths = experiment_config.resolve_project_paths().ensure_artifact_dirs()
config = experiment_config.build_dataset_preparation_config(paths)
paths


In [ ]:
metadata = load_metadata(config.metadata_path)
dicom_mapping = build_dicom_mapping(config.dicom_dir)

print(f"Filtered metadata rows: {len(metadata)}")
print(f"Mapped DICOM files: {len(dicom_mapping)}")
metadata.head()

In [ ]:
sample_row = metadata.sample(1, random_state=42).iloc[0]
dicom_name = dicom_mapping[sample_row['File Name']]
image, _ = normalize_and_resize_dicom(config.dicom_dir / dicom_name, config.resize_dim)
augment = create_augmentation_pipeline()

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Normalized')
axes[0].axis('off')

for index in range(3):
    augmented = augment(image=image)['image']
    axes[index + 1].imshow(augmented, cmap='gray')
    axes[index + 1].set_title(f'Augmented {index + 1}')
    axes[index + 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
artifacts = prepare_dataset(config)
artifacts

In [ ]:
train_df = pd.read_csv(artifacts.train_split_path)
sample_paths = train_df['image_path'].head(5).tolist()

fig, axes = plt.subplots(1, len(sample_paths), figsize=(15, 4))
for axis, image_path in zip(axes, sample_paths):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    axis.imshow(image, cmap='gray')
    axis.set_title(Path(image_path).name)
    axis.axis('off')

plt.tight_layout()
plt.show()